In [10]:
# ============================================================
# INGEST RACES — LOCAL EXECUTION
# ============================================================

import os
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
import nbformat
from nbconvert import PythonExporter

In [11]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb")

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [12]:
# -----------------------------------------
# 2. Detect latest landing batch folder
# -----------------------------------------
batch_folders = sorted([
    f for f in os.listdir(LANDING_PATH)
    if os.path.isdir(os.path.join(LANDING_PATH, f))
])

if not batch_folders:
    raise Exception("❌ No batch folders found in landing!")

p_batch_id = batch_folders[-1]
print("Detected landing batch folder:", p_batch_id)

BATCH_LANDING_PATH = f"{LANDING_PATH}/{p_batch_id}"

Detected landing batch folder: 2025-01


In [13]:
# -----------------------------------------
# 3. Generate pipeline batch_id
# -----------------------------------------
batch_id = generate_batch_id()
print("Pipeline batch_id:", batch_id)

Pipeline batch_id: 20260609_131145


In [14]:
# -----------------------------------------
# 4. Define source + target paths
# -----------------------------------------
source_file = f"{BATCH_LANDING_PATH}/races.csv"
target_path = f"{BRONZE_PATH}/races"
os.makedirs(target_path, exist_ok=True)

print("Source file:", source_file)
print("Target path:", target_path)

Source file: /Users/manoharazalki/F1-Analytics/data/landing/2025-01/races.csv
Target path: /Users/manoharazalki/F1-Analytics/data/bronze/races


In [15]:
# -----------------------------------------
# 5. Define schema
# -----------------------------------------
races_schema = StructType([
    StructField('season', IntegerType(), True),
    StructField('round', IntegerType(), True),
    StructField('url', StringType(), True),
    StructField('raceName', StringType(), True),
    StructField('date', DateType(), True),
    StructField('circuitId', StringType(), True)
])

In [16]:
# -----------------------------------------
# 6. Read CSV
# -----------------------------------------
races_df = (
    spark.read
        .format("csv")
        .schema(races_schema)
        .option("header", True)
        .option("mode", "FAILFAST")
        .load(source_file)
)

print("✔ Races file read successfully")
races_df.show(5)

✔ Races file read successfully
+------+-----+--------------------+------------------+----------+------------+
|season|round|                 url|          raceName|      date|   circuitId|
+------+-----+--------------------+------------------+----------+------------+
|  1950|    1|https://en.wikipe...|british grand prix|1950-05-13| silverstone|
|  1950|    2|https://en.wikipe...| monaco grand prix|1950-05-21|      monaco|
|  1950|    3|https://en.wikipe...|  indianapolis 500|1950-05-30|indianapolis|
|  1950|    4|https://en.wikipe...|  swiss grand prix|1950-06-04|  bremgarten|
|  1950|    5|https://en.wikipe...|belgian grand prix|1950-06-18|         spa|
+------+-----+--------------------+------------------+----------+------------+
only showing top 5 rows


In [17]:
# -----------------------------------------
# 7. Add ingestion metadata
# -----------------------------------------
races_final_df = add_ingestion_metadata(races_df, source_file)

print("✔ Metadata added")
races_final_df.show(5)

✔ Metadata added
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+
|season|round|                 url|          raceName|      date|   circuitId|    ingest_timestamp|         source_file|
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+
|  1950|    1|https://en.wikipe...|british grand prix|1950-05-13| silverstone|2026-06-09 13:11:...|/Users/manoharaza...|
|  1950|    2|https://en.wikipe...| monaco grand prix|1950-05-21|      monaco|2026-06-09 13:11:...|/Users/manoharaza...|
|  1950|    3|https://en.wikipe...|  indianapolis 500|1950-05-30|indianapolis|2026-06-09 13:11:...|/Users/manoharaza...|
|  1950|    4|https://en.wikipe...|  swiss grand prix|1950-06-04|  bremgarten|2026-06-09 13:11:...|/Users/manoharaza...|
|  1950|    5|https://en.wikipe...|belgian grand prix|1950-06-18|         spa|2026-06-09 13:11:...|/Users/manoharaza...|
+------+-----+-

In [18]:
# -----------------------------------------
# 8. Write to Bronze (partitioned by batch_id)
# -----------------------------------------
write_to_bronze(
    races_final_df,
    f"{target_path}/data.parquet",
    batch_id
)

print(f"✔ Races Bronze written to: {target_path}/data.parquet")


✔ Races Bronze written to: /Users/manoharazalki/F1-Analytics/data/bronze/races/data.parquet


In [19]:
# -----------------------------------------
# 9. Read back for validation
# -----------------------------------------
spark.read.parquet(f"{target_path}/data.parquet").show(5)

+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+---------------+
|season|round|                 url|          raceName|      date|   circuitId|    ingest_timestamp|         source_file|       batch_id|
+------+-----+--------------------+------------------+----------+------------+--------------------+--------------------+---------------+
|  1950|    1|https://en.wikipe...|british grand prix|1950-05-13| silverstone|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|
|  1950|    2|https://en.wikipe...| monaco grand prix|1950-05-21|      monaco|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|
|  1950|    3|https://en.wikipe...|  indianapolis 500|1950-05-30|indianapolis|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|
|  1950|    4|https://en.wikipe...|  swiss grand prix|1950-06-04|  bremgarten|2026-06-09 13:11:...|/Users/manoharaza...|20260609_131145|
|  1950|    5|https://en.wikipe...|belgia